(class2-ex-pipeline)=
# Exercise 3. Classification Pipeline
Throughout the previous exercises, we have been performing several steps: 
```{figure} ../figures/class2/ml-pipeline.png
---
name: ml-pipeline
---
```
We can implement these steps as functions, making it easy to modify steps like data processing. This modularity also boosts reproducibility!

In [1]:
## CODE CHUNK REMOVED FOR USERS - HERE TO RELOAD DATA ##
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split

## LOAD DATA ## 
# path of notebook
path = Path.cwd()

data_path = path.parents[1] / "resources" / "data" / "raid" / "train_none.csv"

raw_df = pd.read_csv(data_path)

## SUBSET DATA ##
df = raw_df[raw_df["model"].isin(["human", "cohere"])]

df["is_human"] = df["model"].apply(lambda x: 1 if x == "human" else 0)

## SPLIT DATA ##
train_df, val_df= train_test_split(
                                                    df,
                                                    test_size=0.20,
                                                    random_state=42,
                                                    stratify=df["is_human"]
                                                    )

/var/folders/gg/gk923hkx2w3bw72pk2shplydry9j0b/T/ipykernel_30086/1245980808.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["is_human"] = df["model"].apply(lambda x: 1 if x == "human" else 0)


## 3.1 Starting Point
At this point, you should have *seen* or played with all the coded needed for the entire pipeline. You should also have defined a `vectorize` function (in exercise 2).

Now it is your turn for the rest of the functions! If the HANDS-ON seems like a lot, try to read only one task at a time and complete that (or get me to help explain!).

### YOUR TURN: Creating the Pipeline
Let's create our own classification pipeline!
:::{admonition} HANDS-ON
:class: red

**TASK 1: Data**  
Create functions for the data steps, including:  
- Subsetting the `raw_df` on `model` to include `human` and a LLM of your choice (e.g., `cohere`, `chatgpt` or a third option)  
- Adding a numerical label variable  
- Creating training splits using `train_test_split` (you can either define it as part of another data function, or just use it as part of task 3!";

**TASK 2: Classification**  
Create functions for the classification steps, including:  
- A function that instantiates a `clf = LogisticRegression()`, fits it on training data, and returns the fitted `clf`
- An `evaluate` function which takes the `fitted` clf & computes predictions. Should return both predictions `y_pred` and the classification `report`  

**TASK 3: Play around!**  
Experiment with your pipeline. Use your functions in a chunk! Here are some ideas:
- Try different vectorization strategies  
- Adjust `max_features`  
- Test different model subsets (e.g., `chatgpt`, `llama-chat`)  
- Compare performance across variations  (e.g., print the report)

If you want to be fancy, you could create a `for loop` that firstly loops over models (and subsets) *then* loops over *vec_type*. This way, you'll only need to write the code once but can test on different variations!
:::

#### Solutions
Task 1: Data Functions

In [2]:
# NO NEED TO IMPORT AGAIN, BUT HERE FOR COHERENCY ##
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from typing import Literal
import pandas as pd

def subset_df_binary(
                    raw_df, 
                    class1: str = "human", 
                    class2: str = "cohere", 
                    class_colname:str = "model", 
                    numerical_label_colname:str=None) -> pd.DataFrame:
    """
    Subset df to a binary class. 
    If 'numerical_label_colname' is set to a string (e.g, "is_human") then return df with that numerical col name, assigning 1 to class 1 and 0 to class 2.

    Defaults to "human" and "cohere"
    """
    df = raw_df[raw_df[class_colname].isin([class1, class2])].copy()

    if numerical_label_colname:
        # assign 1 to class1 (so the same as if x == "human" in our example, here we just customize it)
        df[numerical_label_colname] = df[class_colname].apply(lambda x: 1 if x == class1 else 0)

    return df 

### COPY-PASTED FROM ABOVE JUST TO GIVE COHERENCY ##
def vectorize(X_train: pd.Series, X_val: pd.Series, vec_type:Literal["bow", "tf-idf"], max_features:int=500):
    """
    Function to vectorize train and val data! 
    """
    if vec_type == "bow":
        vectorizer = CountVectorizer(lowercase=True, max_features=max_features)
    elif vec_type== "tf-idf":
        vectorizer = TfidfVectorizer(lowercase=True, max_features=max_features)
    else: 
        # this is good code practice, but if your function has no 'else' statement, this is also fine!
        raise ValueError(f"Invalid vec_type: {vec_type}. Must be either 'bow' or 'tf-idf")
    
    X_train_vectorized = vectorizer.fit_transform(X_train)
    X_val_vectorized = vectorizer.transform(X_val)

    return X_train_vectorized, X_val_vectorized

Task 2: CLF functions.

In [3]:
# NO NEED TO IMPORT AGAIN, BUT HERE FOR COHERENCY ##
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

def clf_fit(X_train, y_train, random_state=42):
    """
    Fit a classifier on X and Y

    Could for example be X_train = X_train_bow and y_train = train_df['is_human']
    """
    clf = LogisticRegression(
    random_state=random_state,
    solver="liblinear",   # better for small/medium sparse datasets
    max_iter=1000,        # allow more iterations
    C=1.0,                # adjust if needed (smaller values = stronger regularization)
    )

    clf.fit(X_train, y_train)

    return clf

def clf_evaluate(clf_fitted, X_val, y_val):
    """
    Evaluate fitted classifier, extracting y_pred from X_val, and creating a classification report
    """
    y_pred = clf_fitted.predict(X_val)

    report = classification_report(y_val, y_pred)

    return report, y_pred 

TASK 3: Experiment

In [4]:
for model in ["cohere", "chatgpt"]:
    # prepare dataset
    df = subset_df_binary(raw_df, class1="human", class2=model, numerical_label_colname="is_human")

    # split
    train_df, val_df = train_test_split(
        df,
        test_size=0.20,
        random_state=42,
        stratify=df["is_human"]
    )

    # vectorize and train/evaluate
    for vec_type in ["bow", "tf-idf"]:
        X_train_vec, X_val_vec = vectorize(
            X_train=train_df["generation"],
            X_val=val_df["generation"],
            vec_type=vec_type,
            max_features=500
        )

        clf = clf_fit(X_train_vec, train_df["is_human"])
        report, y_pred = clf_evaluate(clf, X_val_vec, val_df["is_human"])
        print(f"Model: {model} | Vectorization type: {vec_type}")
        print(report)

Model: cohere | Vectorization type: bow
              precision    recall  f1-score   support

           0       0.80      0.90      0.85      5349
           1       0.74      0.55      0.63      2674

    accuracy                           0.78      8023
   macro avg       0.77      0.73      0.74      8023
weighted avg       0.78      0.78      0.78      8023

Model: cohere | Vectorization type: tf-idf
              precision    recall  f1-score   support

           0       0.80      0.91      0.85      5349
           1       0.74      0.55      0.63      2674

    accuracy                           0.79      8023
   macro avg       0.77      0.73      0.74      8023
weighted avg       0.78      0.79      0.78      8023

Model: chatgpt | Vectorization type: bow
              precision    recall  f1-score   support

           0       0.97      0.97      0.97      5349
           1       0.94      0.94      0.94      2674

    accuracy                           0.96      8023
   m

## 3.2 Future Work. Accuracy by Domains 
I won't make an exercise for this due to time, but it *could* be interesting to examine the *accuracy* by domain in the `raid` data. 

If you are curious about this, you should be able to compute it per domain using `y_pred` and `y_true`, for example with [scikit-learn’s `accuracy_score`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.accuracy_score.html).